# पाठ १८: क्रिप्टोग्राफिक रसीदहरू प्रयोग गरी एआई एजेन्टहरूलाई सुरक्षित बनाउने

## व्यावहारिक नोटबुक

यस नोटबुकले चार अभ्यासहरू मार्फत हिँड्छ:

१. एजेन्ट टुल कलको लागि **आफ्नो पहिलो रसीदमा हस्ताक्षर गर्नुहोस्** र त्यसलाई प्रमाणीकरण गर्नुहोस्।
२. **रसीदसँग छेड़छाड़ गर्नुहोस्** र प्रमाणीकरण असफल हुँदै गएको हेर्नुहोस्।
३. **तीन-रसीद श्रृंखला निर्माण गर्नुहोस्** र श्रृंखलाको अखण्डता पुष्टि गर्नुहोस्।
४. **माइक्रोसफ्ट एजेन्ट फ्रेमवर्क टुल कललाई र्याप गर्नुहोस्** ताकि हरेक क्रियाले रसीद निकालोस्।

सबै क्रिप्टोग्राफिक प्रिमिटिभहरू राम्रोसँग मर्मत गरिएका पुस्तकालयहरूबाट आयात गरिएका छन् (`pynacl` Ed25519 का लागि, `jcs` RFC 8785 क्यानोनिकल JSON का लागि, `hashlib` पायथन स्ट्यान्डर्ड लाइब्रेरीबाट SHA-256 का लागि)। रसीद तर्क आफैं तपाईंले पढ्न र संशोधन गर्न सक्ने सादा पायथन हो।

कोषहरू क्रमिक रूपमा चलाउनुहोस्। प्रत्येक खण्ड छोटो र स्व-सम्पूर्ण छ।


## सेटअप

दुई निर्भरताहरू स्थापना गर्नुहोस्। दुवैसँग अनुमति दिने लाइसेन्सहरू छन् (Apache-2.0 / MIT)।


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## सहायक उपयोगिताहरू

यी दुई सहायकहरूले आधार64url एन्कोडिङ (प्याडिङ बिना) र मनग्य वस्तुहरूको SHA-256 ह्यासिङ सम्हाल्छन्। तिनीहरूले बाँकी नोटबुकलाई रसिद तर्कमा केन्द्रित राख्छन्।


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: तपाइँको पहिलो रसीदमा हस्ताक्षर गर्नुहोस्

कल्पना गर्नुहोस् हाम्रो **Contoso Travel** को एजेन्टले हालै सिड्नीबाट लस एन्जलससम्मको उडान ग्राहकको लागि खोज्यो। हामी यो उपकरण कललाई हस्ताक्षर गरिएको रसीदको रूपमा रेकर्ड गर्न चाहन्छौं ताकि भविष्यको अडिटर यसलाई हामीमाथि विश्वास नगरी प्रमाणित गर्न सकोस्।

### Step 1.1: हस्ताक्षर कुञ्जी जेनेरेट गर्नुहोस्

उत्पादनमा, एजेन्टको हस्ताक्षर कुञ्जी हार्डवेयर सुरक्षा मोड्युल (HSM), Azure Key Vault, वा समान संरक्षित स्टोरमा बस्नेछ। यस पाठका लागि हामी नयाँ कुञ्जी स्मृतिमा जेनेरेट गर्छौं।


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: रिसिप्ट पेलोड निर्माण गर्नुहोस्

पेलोडले हामीले रिसिप्टले प्रमाणित गर्न चाहने सबै कुरा समावेश गर्दछ: कुनले कार्य गर्यो, कुन उपकरणले, कुन arguments सँग, के फर्क्यो, कुन नीति अन्तर्गत, र कहिले। हामीले arguments र नतिजालाई सिधै समावेश नगरी हैश गर्छौं ताकि रिसिप्टले संवेदनशील सामग्री चुहाउन नदियोस्।


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: रसिदमा हस्ताक्षर गर्नुहोस् र जम्मा गर्नुहोस्

तीन चरणहरू:

1. JCS प्रयोग गरी प्यालोडलाई कैनोनिकलाइज गर्नुहोस् जसले समान तार्किक रसिद उत्पादन गर्ने दुई कार्यान्वयनहरूले बाइट-परिचयात्मक बाइटहरू उत्पादन गर्छन्।
2. कैनोनिकल बाइटहरूलाई SHA-256 सँग ह्यास गर्नुहोस्।
3. Ed25519 निजी कुञ्जीले ह्यासमा हस्ताक्षर गर्नुहोस्।

पछि हस्ताक्षरलाई मूल प्यालोडमा जोडेर अन्तिम रसिद उत्पादन गरिन्छ।


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### चरण 1.4: प्राप्तिको पुष्टि गर्नुहोस्

पुष्टि प्रक्रियाले यसलाई उल्टो गर्दछ। हामी हस्ताक्षर हटाउँछौं, क्यानोनिकल ह्यास पुनः गणना गर्छौं, र हस्ताक्षर प्राप्तिमा रहेको सार्वजनिक कुञ्जीसँग मिलाउँछौं।

यो पुष्टि गर्ने अडिटरलाई हामीबाट केहि आवश्यक पर्दैन केवल प्राप्ति स्वयं। कल गर्नको लागि कुनै सेवा छैन, सोध्ने कुनै कुञ्जी निर्देशिका छैन, विश्वास गर्नुपर्ने कुनै कुरा छैन।


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

तपाईंले `Receipt is valid: True` देख्नु पर्नेछ। एजेन्टले यसको पहिलो क्रिप्टोग्राफिक रूपमा हस्ताक्षर गरिएको लेखा परीक्षण अभिलेख बनाएको छ।


## Section 2: रसिदसँग छेडछाड गर्नुहोस्

रसिदहरूको मुख्य उद्देश्य तिनीहरू छेडछाड-प्रमाणित हुनु हो। अब यो प्रमाणित गरौं।

हामी रसिदको ठीक एकवटा अक्षर मात्र संशोधन गर्नेछौं र प्रमाणीकरण असफल हुँदै गएर हेर्नेछौं।


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### के भयो ठीकै अब?

जब हामीले `policy_id` परिवर्तन गर्यौं, क्यानोनिकल बाइटहरू परिवर्तन भए। ती बाइटहरूको SHA-256 ह्याश परिवर्तन भयो। हस्ताक्षर (जो मूल ह्याशमाथि थियो) अब नयाँ ह्याशसँग मेल खाँदैन। प्रमाणीकरणले सही रूपमा `False` फिर्ता दिन्छ।

रिसिप्टको कुनै पनि फिल्ड परिमार्जन गरेर प्रमाणीकरण गराउन सकिन्न, जबसम्म आक्रमणकारीसँग निजी कुञ्जी हुँदैन। निजी कुञ्जी कुंजी संग्रहमा रहँदा र सार्वजनिक कुञ्जी प्रकाशित हुँदा, छेडछाड लुकाउन असम्भव हुन्छ।

आफ्नै प्रयास गर्नुहोस्: माथिको सेलमा रहेको `tool_name` वा `agent_id` वा `timestamp` परिमार्जन गरेर पुन: चलाउनुहोस्। प्रत्येक परिवर्तनले अमान्य रिसिप्ट उत्पादन गर्छ।


## Section 3: चेन प्राप्तिहरूलाई सँगै जोड्नुहोस्

एउटा मात्र प्राप्तिले एउटा क्रियालाई सुरक्षा गर्छ। अधिकांश एजेन्टहरूले धेरै क्रियाहरू गर्छन्। सम्पूर्ण अनुक्रमलाई ट्याम्पर-एभिडेन्ट बनाउनका लागि, हामी प्रत्येक प्राप्तिलाई अघिल्लो प्राप्तिसँग जोड्छौँ जसले नयाँ प्राप्तिको पेलोडमा अघिल्लो प्राप्तिको ह्यास समावेश गर्छ।

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

यदि कोहीले कुनै प्राप्ति हटाउँछ वा पुनः क्रमबद्ध गर्छ भने, चेन ठीक त्यहि बिन्दुमा टुट्छ। कुनै पनि पछि आउने प्राप्तिको प्रमाणिकरण असफल हुन्छ किनभने यसको `previous_receipt_hash` अब यसको पूर्ववर्तीको वास्तविक ह्याससँग मेल खाँदैन।


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

अब मध्यको रसीदलाई छेडछाड गरेर श्रृंखला तोड्नुहोस् र पुनः प्रमाणीकरण गर्नुहोस्। छेडछाड गरिएको रसीद यसको हस्ताक्षर जाँचमा असफल हुन्छ, र अर्को रसीद यसको श्रृंखला-जडान जाँचमा असफल हुन्छ (किनभने यसको `previous_receipt_hash` अब संशोधित मध्य रसीदको ह्याससँग मेल खाँदैन)।


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 अझै प्रमाणित हुन्छ (यो परिवर्तन गरिएको थिएन र निर्भर गर्न कुनै पूर्वसन्धि थिएन)। Receipt 1 ले आफ्नो हस्ताक्षर जाँच असफल गर्यो किनकि हामीले `tool_args_hash` परिवर्तन गर्‍यौँ। Receipt 2 ले यसको चेन-लिंक जाँच असफल गर्यो किनकि यसको `previous_receipt_hash` मूल (अब परिवर्तन गरिएको) receipt 1 विरुद्ध गणना गरिएको थियो।

यदि एक आक्रमणकारीले परिवर्तन गरिएको receipt 1 पुनः-हस्ताक्षर गर्छ (जो उनीहरूले निजी कुञ्जी बिना सक्दैनन्), receipt 2 मा चेन-लिंक असन्तुलनले पनि हेरफेर देखाउनेछ। परिवर्तन लुकाउनका लागि, आक्रमणकारीले परिवर्तनको बिन्दुबाट सुरु भएर प्रत्येक receipt पुनः-हस्ताक्षर गर्नुपर्ने हुन्छ, जसका लागि निजी कुञ्जीको धारण आवश्यक पर्छ।


## खण्ड ४: रसीद हस्ताक्षर सहित एजेन्ट टूल कललाई र्याप गर्नुहोस्

व्यावहारिक सञ्चालनमा, तपाईं प्रत्येक एजेन्ट लेखकलाई `make_receipt` कल गर्न सम्झन नचाहनुहुन्छ। तपाईं प्रत्येक टूल कलमा रसीद हस्ताक्षर स्वचालित हुन चाहनुहुन्छ।

यहाँ सबैभन्दा सरल ढाँचा छ: एउटा र्यापर वर्ग जुन कुनै पनि कलयोग्य टूल फंक्शनलाई लिन्छ र यसको रसीद-निष्पादन गर्ने संस्करण फर्काउँछ। यसले कुनै पनि एजेन्ट फ्रेमवर्कलाई अनुकूल बनाउँछ, जस्तै Microsoft Agent Framework (`agent_framework.azure`)।

यदि तपाईंको Azure AI Foundry परियोजना सेटअप छैन भने पनि, तलको स्थानीय मोकले यस ढाँचालाई प्रदर्शन गर्छ।


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Microsoft Agent Framework सँग एकीकरण गर्नुहोस्

माथिको `ReceiptedTool` रैपर फ्रेमवर्क-स्वतन्त्र छ। Microsoft Agent Framework सँग निर्माण गरिएको एजेन्ट भित्र यसलाई प्रयोग गर्न, र्याप गरिएको फंक्शनलाई टूलको रूपमा दर्ता गर्नुहोस्। एउटा स्केच (तपाईंले नकलीलाई वास्तविक Azure AI Foundry टूल दर्तासँग परिवर्तन गर्नुहुनेछ):

```python
# समाकलन आकृति देखाउने स्यूडोकोड।
# from agent_framework.azure बाट AzureAIProjectAgentProvider आयात गर्दै
# from azure.identity बाट AzureCliCredential आयात गर्दै
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     निर्देशनहरू="तपाईं एक Contoso यात्रा एजेन्ट हुनुहुन्छ ..."
#     उपकरणहरू=[receipted_lookup],   # आवृत्त गरिएको उपकरण, कच्चा функ्सन होइन
# )
# प्रतिक्रिया = agent.run("जुन महिनामा Sydney बाट Los Angeles का लागि उडानहरू फेला पार्नुहोस्।")
#
# # रन पछि, एजेन्टले गरेका हरेक उपकरण कलसँग एक हस्ताक्षरित रसिद हुन्छ:
# audit_chain = receipted_lookup.receipts
```

एजेन्ट फ्रेमवर्कले रसीदबारे केही जान्न आवश्यक छैन। रसीदमा हस्ताक्षर टूलको वरिपरि र्याप गरिएको हुन्छ, फ्रेमवर्कमा जोडिएको हुँदैन। यसरी तपाईं एजेन्ट पुनर्लेखन नगरी विद्यमान एजेन्ट कोडमा provenance थप्न सक्नुहुन्छ।


## पुनरावलोकन र चुनौती विस्तार

तपाईंसँग छ:

- Ed25519 कुञ्जी जोडा सिर्जना गरिएको।
- एजेन्ट उपकरण कलका लागि एक रसीद तयार गरी हस्ताक्षर गरिएको।
- रसीदलाई केवल सार्वजनिक कुञ्जी प्रयोग गरेर अफलाइन जाँच गरिएको।
- रसीदलाई छेडछाड गरी जाँच असफल भएको देखिएको।
- तीन रसीदहरूको ह्यास-चेन श्रृंखला तयार गरिएको।
- श्रृंखलाको मध्य भागमा छेडछाड गरी दुवै हस्ताक्षर असफलता र श्रृंखला-लक असफलता देखिएको।
- एक उपकरण फङ्सनलाई स्वतः रसीद हस्ताक्षर गर्ने तरिकाले र्याप गरिएको।

**चुनौती विस्तार।** रसीद स्किमामा `request_id` क्षेत्र (वितरित ट्रेसिङका लागि UUID) थप्नुहोस्। `make_receipt` मा यसलाई समावेश गर्न अपडेट गर्नुहोस्, र पक्का गर्नुहोस् कि रसीदहरू अन्त्य देखि अन्त सम्म सत्यापित हुन्छन्। त्यसपछि हस्ताक्षर पछि यो क्षेत्र परिवर्तन गरी जाँच असफल हुन्छ भनेर पुष्टि गर्नुहोस्। यसले तपाईंलाई क्यानोनिकल इन्कोडिंगको हरेक बाइटले हस्ताक्षरतर्फ कसरी योगदान पुर्‍याउँछ भनी आन्तरिक रूपमा बुझ्न बाध्य पार्दछ।

**महत्वपूर्ण सीमा।** रसीदहरूले तीन कुरा मात्र प्रमाणित गर्दछन्: नियुक्ति (यो कुञ्जीले यो सामग्रीमा हस्ताक्षर गर्‍यो), सम्पेक्षता (सामग्री हस्ताक्षरपछि परिवर्तन भएको छैन), र क्रमबद्धता (यो रसीद त्यो रसीदपछि आएको हो)। तिनीहरूले प्रमाणित गर्दैनन् कि एजेन्टको क्रिया सही थियो, `policy_id` मा नामकरण गरिएको नीति साँच्चिकै मूल्यांकन गरिएको थियो, वा एजेन्टले सबै नियमहरू पालना गर्‍यो। रसीदहरू आधार हुन्। शासन तपाईँले माथि बनाउने प्रणाली हो।

त्यो सीमालाई ध्यानमा राख्दै पाठ्यक्रम README फेरि पढ्नुहोस्। सबैभन्दा सामान्य गल्ती भनेको "हामीसँग रसीदहरू छन्" भन्ने कुरा "हामी शासित छौँ" भन्नु हो। त्यो होइन। रसीदहरूले एजेन्टको व्यवहार परीक्षणयोग्य बनाउँछन्। तिनीहरूले यसलाई सही बनाउँदैनन्।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
